# 02 — CSP — load, edit, save

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/02_csp_io.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/02_csp_io.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/02_csp_io.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Build, edit and round-trip the seven-parameter CSP file format that the Parameters tab consumes.

**Mirrors.** **Parameters** tab → *Load CSV*, *Edit table*, *Download CSV*.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Why CSP I/O matters

The CSP (Cube Specification) is the seven-parameter probabilistic input vector for the model. Every result in the thesis is conditional on a CSP, so being able to load, edit, save and reload one is a prerequisite for reproducibility.

The Parameters tab in the dashboard exposes three operations:
1. **Load CSV** — replace the running CSP with one from disk.
2. **Edit table** — change individual `mean` / `sd` cells.
3. **Download CSV** — round-trip back to disk.

This notebook does the same three operations from Python.


In [ ]:
from cubespec import DEFAULT_CSP, load_csp_csv, save_csp_csv
from pathlib import Path
import pandas as pd

csp = DEFAULT_CSP
print('Default CSP')
print(pd.DataFrame({'mean': csp.means(), 'sd': csp.sds()}, index=csp.keys()))


## 2. Round-trip to disk

Save then reload — the result must equal the original.


In [ ]:
tmp = Path('/tmp/csp_roundtrip.csv')
save_csp_csv(csp, tmp)
csp2 = load_csp_csv(tmp)
import numpy as np
assert np.allclose(csp.means(), csp2.means())
assert np.allclose(csp.sds(), csp2.sds())
print('round-trip OK ·', tmp.read_text().splitlines()[0])
print(tmp.read_text())


## 3. Edit a cell — mirrors the dashboard's editable table

Bumping the mean of `P4_Fx` (axial load) by +50 kN should shift the predicted P9 mean. We compute both, side-by-side.


In [ ]:
from copy import deepcopy
from cubespec import sample_independent, compute_outputs_batch

edited = deepcopy(csp)
edited.params['P4_Fx'].mean += 50_000  # +50 kN

X0 = sample_independent(csp,    n=20_000, seed=1337)
X1 = sample_independent(edited, n=20_000, seed=1337)
Y0 = compute_outputs_batch(X0)[:, 2]
Y1 = compute_outputs_batch(X1)[:, 2]
print(f'P9 mean (default)   = {Y0.mean():.3f} MPa')
print(f'P9 mean (P4 +50 kN) = {Y1.mean():.3f} MPa')
print(f'Δ                   = {Y1.mean() - Y0.mean():+.3f} MPa')


## 4. Visualise the shift


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(Y0, bins=50, alpha=0.55, color='#5b8def', label='default CSP')
ax.hist(Y1, bins=50, alpha=0.55, color='#e94f64', label='P4_Fx mean +50 kN')
ax.set_xlabel('P9 compressive strength (MPa)')
ax.set_ylabel('count')
ax.set_title('Effect of editing one CSP cell on the predicted P9 distribution')
ax.legend()
fig.tight_layout()
plt.show()


## 5. Try this

Edit `csp.params['P0_rho'].sd` (mass density standard deviation) instead and observe how the **width** of the P9 distribution changes — that is the signature of an aleatoric input.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
